# Heart Disease: Data Acquisition & EDA

This notebook acquires the Cleveland subset of the UCI Heart Disease dataset, audits its quality, creates a cleaned and one-hot encoded CSV, and saves three EDA figures.

**Target definition:** `0` means no disease; original target values `1-4` are mapped to `1` (disease present). This follows the dataset's documented disease-severity field and the assignment's binary-classification objective.

Run from a clean setup after installing `requirements.txt`. The acquisition cell downloads the source when the raw file is absent or empty. The preprocessing CSV is for this assignment's EDA handoff; for later model evaluation, fit imputers and encoders on training folds only to avoid data leakage.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data_preprocessing.py").is_file():
            return candidate
    nested_project = start / "mlops_heart_disease"
    if (nested_project / "src" / "data_preprocessing.py").is_file():
        return nested_project
    raise FileNotFoundError("Open the project folder or run this notebook from its parent folder.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_acquisition import download_dataset
from src.data_preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    clean_data,
    encode_features,
    read_raw_data,
    save_processed_data,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "heart_disease_clean.csv"
FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
print(f"Project root: {PROJECT_ROOT}")

## 1. Acquisition and data dictionary

The raw file is the processed Cleveland subset from the [UCI Heart Disease repository](https://archive.ics.uci.edu/dataset/45/heart+disease). The repository distributes this 14-column file without a header. The downloader checks its row count, column count, and disease-severity values before writing it to `data/raw/processed.cleveland.data`.

The attributes are age, sex, chest-pain type (`cp`), resting blood pressure (`trestbps`), serum cholesterol (`chol`), fasting blood sugar (`fbs`), resting ECG (`restecg`), maximum heart rate (`thalach`), exercise-induced angina (`exang`), ST depression (`oldpeak`), slope, number of major vessels (`ca`), thalassemia (`thal`), and disease severity (`num`). The source encodes unknown feature values as `?`.

In [ ]:
if not RAW_PATH.exists() or RAW_PATH.stat().st_size == 0:
    print(f"Downloading dataset from UCI to {RAW_PATH} ...")
    download_dataset(RAW_PATH)
else:
    print(f"Using existing raw dataset: {RAW_PATH}")

raw_data = read_raw_data(RAW_PATH)
assert raw_data.shape[1] == 14, f"Expected 14 columns, found {raw_data.shape[1]}"
print(f"Loaded {len(raw_data):,} rows and {raw_data.shape[1]} columns.")

## 2. Initial quality audit

Audit missing values and the original five-level target before transforming anything. The original target levels are severity values; for this assignment, any positive level is grouped as heart disease present.

In [ ]:
display(raw_data.head())

missing_before = raw_data.isna().sum().rename("missing_values")
print("Missing-value count by source attribute:")
display(missing_before[missing_before.gt(0)].to_frame())

print("Original disease-severity distribution:")
display(
    raw_data["num"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("num (original severity)")
    .to_frame("records")
)
print(f"Exact duplicate rows before cleaning: {raw_data.duplicated().sum()}")

## 3. Cleaning and encoding

`?` is parsed as missing. Continuous measurements are imputed with their median; categorical codes use their most frequent observed category. Categorical predictors are one-hot encoded, continuous features remain in their original units, and `num > 0` becomes the binary `target`. Exact duplicate rows are reported above but retained because this step does not establish that repeated measurements are erroneous.

The imputation values here describe the full dataset for the EDA artifact. In the modeling task, fit preprocessing only on each training fold.

In [ ]:
cleaned_data = clean_data(raw_data)
encoded_data = encode_features(cleaned_data)
processed_path = save_processed_data(encoded_data, PROCESSED_PATH)

assert not cleaned_data.isna().any().any(), "Missing values remain after cleaning"
assert encoded_data.select_dtypes(exclude="number").empty, "Encoded data contains nonnumeric columns"
assert set(encoded_data["target"].unique()) == {0, 1}, "Target is not binary"

print(f"Cleaned rows: {len(cleaned_data):,}")
print(f"Encoded predictors: {encoded_data.shape[1] - 1}")
print(f"Saved processed data to: {processed_path}")
display(encoded_data.head())

## 4. Feature distributions

Continuous measurements are shown as histograms; categorical attributes are shown as category-frequency distributions so their discrete codes are not misrepresented as continuous measurements. Both the cleaned patient-level data and the target are used before one-hot expansion.

In [ ]:
feature_labels = {
    "age": "Age (years)",
    "sex": "Sex code",
    "cp": "Chest-pain type",
    "trestbps": "Resting blood pressure (mm Hg)",
    "chol": "Serum cholesterol (mg/dL)",
    "fbs": "Fasting blood sugar > 120 mg/dL",
    "restecg": "Resting ECG code",
    "thalach": "Maximum heart rate (bpm)",
    "exang": "Exercise-induced angina",
    "oldpeak": "ST depression",
    "slope": "ST-slope code",
    "ca": "Major vessels (0-3)",
    "thal": "Thalassemia code",
}
feature_order = NUMERIC_FEATURES + CATEGORICAL_FEATURES
column_count = 4
row_count = int(np.ceil(len(feature_order) / column_count))
fig, axes = plt.subplots(row_count, column_count, figsize=(16, 3.8 * row_count))
axes = axes.flatten()

for axis, feature in zip(axes, feature_order):
    if feature in NUMERIC_FEATURES:
        sns.histplot(
            data=cleaned_data,
            x=feature,
            bins=20,
            kde=True,
            color="#367A72",
            ax=axis,
        )
        axis.axvline(cleaned_data[feature].median(), color="#C65F45", linestyle="--", linewidth=1.5)
    else:
        sns.countplot(data=cleaned_data, x=feature, color="#D07B56", ax=axis)
        axis.tick_params(axis="x", rotation=0)
    axis.set_title(feature_labels[feature], fontsize=10, weight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Records")

for axis in axes[len(feature_order):]:
    axis.set_visible(False)

fig.suptitle("Cleveland Heart Disease | Feature Distributions", fontsize=16, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(FIGURES_PATH / "feature_histograms.png", dpi=220, bbox_inches="tight")
plt.show()

## 5. Class balance

The target distribution is reported as both counts and percentages to make any imbalance visible before modeling.

In [ ]:
class_counts = cleaned_data["target"].value_counts().reindex([0, 1], fill_value=0)
class_labels = ["No disease (0)", "Disease present (1)"]
class_percentages = class_counts / class_counts.sum() * 100

fig, axis = plt.subplots(figsize=(7.5, 5))
bars = axis.bar(class_labels, class_counts.values, color=["#367A72", "#D07B56"], width=0.62)
for bar, count, percentage in zip(bars, class_counts.values, class_percentages.values):
    axis.annotate(
        f"{count} ({percentage:.1f}%)",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=11,
        weight="bold",
    )
axis.set_title("Binary Target Class Balance", fontsize=14, weight="bold")
axis.set_ylabel("Patients")
axis.set_xlabel("")
axis.set_ylim(0, class_counts.max() * 1.16)
fig.tight_layout()
fig.savefig(FIGURES_PATH / "class_balance.png", dpi=220, bbox_inches="tight")
plt.show()

display(
    pd.DataFrame(
        {"records": class_counts.to_numpy(), "percentage": class_percentages.to_numpy().round(2)},
        index=pd.Index(class_labels, name="target"),
    )
)

## 6. Correlation structure

Pearson correlations are computed over continuous measurements, one-hot categorical indicators, and the binary target. Correlations among encoded category indicators are expected (for example, mutually exclusive categories); they should not be interpreted as causal relationships.

In [ ]:
correlation = encoded_data.corr(method="pearson")
upper_triangle = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
fig, axis = plt.subplots(figsize=(17, 14))
sns.heatmap(
    correlation,
    mask=upper_triangle,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.2,
    cbar_kws={"shrink": 0.72, "label": "Pearson correlation"},
    ax=axis,
)
axis.set_title("Feature Correlation | Encoded Data", fontsize=15, weight="bold", pad=16)
axis.tick_params(axis="x", labelrotation=55, labelsize=8)
axis.tick_params(axis="y", labelrotation=0, labelsize=8)
fig.tight_layout()
fig.savefig(FIGURES_PATH / "correlation_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

## 7. EDA summary and handoff

Use the observed class proportions and the summaries below to guide later modeling decisions. This notebook does not train a model or select features; those belong to the next assignment task.

In [ ]:
print(f"EDA dataset: {len(cleaned_data):,} records, {encoded_data.shape[1] - 1} encoded predictors")
print(f"Missing values after cleaning: {int(cleaned_data.isna().sum().sum())}")
print(f"Disease-present share: {class_percentages.loc[1]:.1f}%")

median_by_class = cleaned_data.groupby("target")[NUMERIC_FEATURES].median().T
median_by_class.columns = ["No disease", "Disease present"]
display(median_by_class)

print("Saved artifacts:")
for artifact_path in (
    PROCESSED_PATH,
    FIGURES_PATH / "class_balance.png",
    FIGURES_PATH / "feature_histograms.png",
    FIGURES_PATH / "correlation_heatmap.png",
):
    print(f"- {artifact_path.relative_to(PROJECT_ROOT)}")